# Data preparation

Assembles the six-asset OHLCV panel (five index/commodity series from Yahoo Finance plus JP Morgan from CRSP), saves the close-to-close price panel `indices.csv`, and computes several realized volatility (RV) estimators. All data paths are managed through `file_manager`.

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from utils.file_manager import quantDiffusionPath

file_manager = quantDiffusionPath()   # all data paths are managed through file_manager

## Helper functions

In [ ]:
def fill_zero_rv_with_sampling(df):
    df_filled = df.copy()

    for col in df.columns:
        zero_mask = (df[col] == 0)
        num_zeros = zero_mask.sum()

        if num_zeros > 0:
            # keep only the non-zero values
            non_zero_values = df.loc[~zero_mask, col].values
            # resample with replacement from the non-zero values
            random_samples = np.random.choice(non_zero_values, size=num_zeros)
            df_filled.loc[zero_mask, col] = random_samples

    print("Filled zero values using each asset's existing distribution.")
    return df_filled


# Compute several realized volatility (RV) estimators
def calculate_rvs(O, H, L, C):
    rv_dict = {}

    # (1) Parkinson (1980) - high-low based
    # Formula: 1 / (4 * ln2) * (ln(H/L))^2
    rv_dict['Parkinson'] = (1 / (4 * np.log(2))) * (np.log(H / L)**2)

    # (2) Garman-Klass (1980) - OHLC based
    # Formula: 0.5 * (ln(H/L))^2 - (2*ln2 - 1) * (ln(C/O))^2
    rv_dict['Garman_Klass'] = 0.5 * (np.log(H / L)**2) - (2 * np.log(2) - 1) * (np.log(C / O)**2)

    # (3) Rogers-Satchell (1991) - drift independent
    # Formula: ln(H/C)*ln(H/O) + ln(L/C)*ln(L/O)
    rv_dict['Rogers_Satchell'] = np.log(H / C) * np.log(H / O) + np.log(L / C) * np.log(L / O)

    # (4) Simple close-to-close volatility (for comparison)
    # squared daily log returns
    returns = np.log(C / C.shift(1))
    rv_dict['Close_to_Close'] = returns**2

    return rv_dict

## 1. Assemble the six-asset panel and save `indices.csv`

In [ ]:
# JPM (CRSP) OHLCV
df_JPM = pd.read_csv(file_manager.get_jpm_file()).set_index('DlyCalDt').sort_index()
df_JPM = df_JPM[['DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol']]
df_JPM.columns = ['O', 'H', 'L', 'C', 'V']

# Combine the raw index OHLCV (data/indices/) with JPM into a six-asset panel
data_dict = {}
for field in ["O", "H", "L", "C", "V"]:
    df = pd.read_csv(file_manager.get_indices_ohlcv_file(field), index_col=0)
    df = df[["^DJI", "^IXIC", "^HSI", "GC=F", "CL=F"]]
    df = pd.concat([df, df_JPM[field]], axis=1)
    df.columns = ["DJI", "NASDAQ", "HSI", "GOLD", "WTI_OIL", "JPM"]
    df = df[["DJI", "NASDAQ", "JPM", "HSI", "GOLD", "WTI_OIL"]]
    df = df.dropna()
    df = df.loc["2001-01-02":"2025-12-31", :]
    df.index = pd.to_datetime(df.index)
    data_dict[field] = df

# Assembled close-price panel = new indices.csv (index closes + JPM)
indices_df = data_dict["C"].copy()
indices_df.index.name = "Date"
indices_df.to_csv(file_manager.get_indices_file())
print(f"saved {file_manager.get_indices_file()}  shape={indices_df.shape}  "
      f"range={indices_df.index.min().date()} ~ {indices_df.index.max().date()}")

## 2. Realized volatility estimators

In [ ]:
# Base OHLC data (from the assembled data_dict)
O = data_dict['O']
H = data_dict['H']
L = data_dict['L']
C = data_dict['C']

# Compute the RV estimators
rv_results = calculate_rvs(O, H, L, C)

In [ ]:
os.makedirs(file_manager.get_data_dir("RV"), exist_ok=True)
for rv_name in rv_results.keys():
    print(rv_name)
    rv_results[rv_name] = fill_zero_rv_with_sampling(rv_results[rv_name])
    rv_results[rv_name].to_csv(file_manager.get_rv_file(rv_name))